<a href="https://colab.research.google.com/github/mf2056/F20AA_CW2/blob/main/Distlbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install transformers[torch] datasets accelerate scikit-learn

In [13]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from transformers import AutoTokenizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from torch import nn

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working on: {device}")

Working on: cuda


In [14]:
train = pd.read_csv("train_model.csv")
test = pd.read_csv("test_model.csv")

In [ ]:
train.head()

,text,rating
0,this place is terrible; the people in charge a...,2
1,terrible service! and they are saying that i n...,1
2,absolutely terrible company. they sent me to c...,1
3,"to find it, either park in front of the tuesda...",4
4,mall location. used their services for sedan. ...,4


In [ ]:
test.head()

,text,rating
0,every time i’ve been hear they’ve done a great...,4
1,if u like calm and layed back without the ster...,4
2,i would give this place 0 if i could. my exper...,1
3,had some problems viewing a place and was not ...,3
4,the staff at this place is very rude! they do ...,1


In [ ]:
train.isnull().sum()

,0
text,0
rating,0


In [ ]:
test.isnull().sum()

,0
text,0
rating,0


In [15]:
train["rating"] = train["rating"] - 1

In [16]:
test["rating"] = test["rating"] - 1

In [17]:
# Calculate balanced weights
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train['rating']),
    y=train['rating']
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(
    train[['text', 'rating']].rename(columns={'rating': 'labels'})
)

test_dataset = Dataset.from_pandas(
    test[['text', 'rating']].rename(columns={'rating': 'labels'})
)


In [18]:

# This will automatically pick the right tokenizer for DistilBERT
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=160
    )

In [19]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/288000 [00:00<?, ? examples/s]

Map:   0%|          | 0/72000 [00:00<?, ? examples/s]

In [20]:
class WeightedTrainer(Trainer):
    # We add 'num_items_in_batch=None' and '**kwargs' to match the new library requirements
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")

        # Standard Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Our custom Weighted Loss logic (stays the same)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro")
    }



In [1]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=5
).to(device)

training_args = TrainingArguments(
    output_dir="./distilbert_final",
    num_train_epochs=2,              # 2 epochs is the sweet spot for 288k rows
    per_device_train_batch_size=32,  # Optimized for 8GB+ VRAM GPUs
    eval_strategy="epoch",           # Evaluate once per hour to save time
    save_strategy="epoch",
    learning_rate=3e-5,
    weight_decay=0.01,
    fp16=True,                       # Enables Mixed Precision (Speed Boost)
    load_best_model_at_end=True,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

NameError: name 'DistilBertForSequenceClassification' is not defined

In [11]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Get predictions
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Per-class metrics
labels = ["1 star", "2 stars", "3 stars", "4 stars", "5 stars"]

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=labels))

# Convert to DataFrame (nice table for report)
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
report_df = pd.DataFrame(report).transpose()

print("\n=== Report as Table ===")
print(report_df)

# Confusion Matrix
print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_true, y_pred)
print(cm)

KeyboardInterrupt: 